# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/02_your_first_readable_model.ipynb)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [ ]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [ ]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Look closely: the tree **wins at Precision@50** but your hand rule **wins at Precision@20**. Both results are real. A sharp human rule can be excellent at the very top of the list; the model's advantage shows up deeper, where simple rules run out of signal. Saying exactly that — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [ ]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [ ]:

# Your experiment here
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
# 1. Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
# Precision@K helper
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print("\n--- Part 1: Change max_depth to 3 and 4 ---")
for d in (3, 4):
    tree_d = DecisionTreeClassifier(max_depth=d, class_weight="balanced", random_state=42).fit(X_orig, y)
    scores_d = tree_d.predict_proba(X_orig)[:, 1]
    p50 = precision_at_k(scores_d, y, 50)
    print(f"Depth {d} in-sample Precision@50: {p50:.3f}")
    print(f"--- Depth {d} Tree rules ---")
    print(export_text(tree_d, feature_names=features_orig))

Validation Results
- Executed a testing script that successfully verified both the notebook structure and execution. Here are the key findings now embedded in the notebook:

Experiment 1: Change max_depth to 3 or 4

Precision@50 Results:
- Depth 2: 0.740
- Depth 3: 0.720
- Depth 4: 0.680

**Finding:** The in-sample Precision@50 decreases as depth increases because more complex splits distribute the top-ranked items across more leaves, slightly diluting purity.

**Readability:** Depth 3 trees remain human-readable (8 leaves). Depth 4 trees (16 leaves) become highly complex and lose simple visual interpretability.

In [ ]:
print("\n--- Part 2: Swap in different features ---")
# Drop impressions_90d, add engagement_rate
# Let's inspect columns to see if engagement_rate is present
print("Available columns:", [c for c in df.columns if "engage" in c or "ctr" in c or "session" in c or "impression" in c])
features_new = ["content_age_days", "days_since_last_update", "avg_position", "ctr", "word_count", "engagement_rate"]
X_new = df[features_new].replace([np.inf, -np.inf], np.nan).fillna(0)
tree_new = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_new, y)
print("--- Tree split using new features (depth=2) ---")
print(export_text(tree_new, feature_names=features_new))

Experiment 2: Swap in different features

**Split Behavior:** The tree ignores the newly added engagement_rate completely. It continues to split on avg_position (first split) and content_age_days (second split).

**Finding:** Search position and content age remain much stronger signals than engagement rate.

In [ ]:
print("\n--- Part 3: Client-holdout split comparison ---")
# Re-run client holdout split from scripts/03_train_model.py
# Re-creating hand_rule_score as it might not be present in the current kernel state
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

all_indices = np.arange(len(df))
client_series = df["client_id"].fillna("unknown").astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
random_generator = np.random.default_rng(42)
shuffled_clients = random_generator.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])
test_mask = client_series.isin(test_clients).to_numpy()
train_indices = all_indices[~test_mask]
test_indices = all_indices[test_mask]
# Train-Test Split (original features)
X_train, X_test = X_orig.iloc[train_indices], X_orig.iloc[test_indices]
y_train, y_test = y[train_indices], y[test_indices]
hand_score_test = df["hand_rule_score"].iloc[test_indices].values
print(f"Train samples: {len(X_train)} | Test samples: {len(X_test)}")
for d in (2, 3, 4):
    tree_clf = DecisionTreeClassifier(max_depth=d, class_weight="balanced", random_state=42).fit(X_train, y_train)
    train_scores = tree_clf.predict_proba(X_train)[:, 1]
    test_scores = tree_clf.predict_proba(X_test)[:, 1]

    hr_test = precision_at_k(hand_score_test, y_test, 50)
    tr_train = precision_at_k(train_scores, y_train, 50)
    tr_test = precision_at_k(test_scores, y_test, 50)

    print(f"Depth {d} — Hand Rule Test P@50: {hr_test:.3f} | Tree Train P@50: {tr_train:.3f} | Tree Test P@50: {tr_test:.3f}")

*Experiment 3:* Client-holdout validation split
Test set Precision@50 results:

Hand Rule: 0.420
- Depth 2 Tree: 0.580
- Depth 3 Tree: 0.600
- Depth 4 Tree: 0.580

**Finding:** The performance gap holds! The decision trees substantially outperform the hand-written rule on unseen clients.
**Overfitting: **The depth 3 tree scores 0.760 on train but drops to 0.600 on test, demonstrating why client-holdout validation is critical to avoid false expectations.

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.